# Composition robustness for the single-task binary endpoint

Repeats the whole pipeline across 20 participant assignments so that the
single-task result can be reported as a distribution rather than as one split.

**Before running:** Runtime -> Change runtime type -> T4 GPU. The dataset
`vrc3po_master_dataset_fixed.csv` must be in `MyDrive`.

Results are written to `MyDrive/vrc3po_composition_robustness/`, one JSON per
composition, as each finishes. A disconnect costs at most one composition.

The compositions share one pool of 84 participants and are **not** independent.
Report the distribution; do not apply a one-sample test against 0.5.

In [ ]:
# ============================================================
# CELL 1 - SETUP.  Run this first, and re-run it after any restart.
# Everything the run loop needs is defined in this one cell.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print('TensorFlow', tf.__version__)

from __future__ import annotations

# ---------------------------------------------------------------------------
# Imports and configuration. Everything the loop needs is defined above it.
# ---------------------------------------------------------------------------
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

FEATURE_COLUMNS = [
    "pupil_diam_L",
    "pupil_diam_R",
    "eye_open_L",
    "eye_open_R",
    "gaze_dir_world_X",
    "gaze_dir_world_Y",
    "gaze_dir_world_Z",
    "gaze_origin_world_X",
    "gaze_origin_world_Y",
    "gaze_origin_world_Z",
    "head_quat_X",
    "head_quat_Y",
    "head_quat_Z",
    "head_quat_W",
]
WINDOW_LENGTH = 30
WINDOW_STRIDE = 15
ELEVATED_THRESHOLD = 2.0
PASSIVE_SOURCES = frozenset({"simulations", "terrain"})
ENSEMBLE_SEEDS = (42, 123, 456, 789, 2024)
BOOTSTRAP_REPLICATES = 5000

DATASET_PATH = Path("/content/drive/MyDrive/vrc3po_master_dataset_fixed.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/vrc3po_composition_robustness")
COMPOSITION_SEEDS = tuple(range(20))


# ---------------------------------------------------------------------------
# Metrics. Reimplemented so the notebook does not depend on sklearn version.
# ---------------------------------------------------------------------------
def roc_auc(y_true, scores):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    positive = np.flatnonzero(y == 1)
    negative = np.flatnonzero(y == 0)
    if not len(positive) or not len(negative):
        return float("nan")
    order = np.argsort(s, kind="mergesort")
    sorted_scores, sorted_y = s[order], y[order]
    concordant = 0.0
    negatives_before = 0.0
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        group_positive = float(np.sum(sorted_y[i:j] == 1))
        group_negative = float(np.sum(sorted_y[i:j] == 0))
        concordant += group_positive * (negatives_before + 0.5 * group_negative)
        negatives_before += group_negative
        i = j
    return float(concordant / (len(positive) * len(negative)))


def average_precision(y_true, scores):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    total_positive = float(np.sum(y == 1))
    if total_positive == 0:
        return float("nan")
    order = np.argsort(-s, kind="mergesort")
    sorted_scores, sorted_y = s[order], y[order]
    true_positive = false_positive = result = 0.0
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        added_positive = float(np.sum(sorted_y[i:j] == 1))
        true_positive += added_positive
        false_positive += float(np.sum(sorted_y[i:j] == 0))
        precision = true_positive / (true_positive + false_positive)
        result += precision * (added_positive / total_positive)
        i = j
    return float(result)


def cluster_bootstrap_auc(y_true, scores, participant_ids, replicates, seed):
    """Resample whole participants, keeping every window of each draw."""
    rng = np.random.default_rng(seed)
    participants = np.unique(participant_ids)
    rows_for = {p: np.flatnonzero(participant_ids == p) for p in participants}
    estimates = []
    for _ in range(replicates):
        sampled = rng.choice(participants, size=len(participants), replace=True)
        rows = np.concatenate([rows_for[p] for p in sampled])
        estimate = roc_auc(y_true[rows], scores[rows])
        if np.isfinite(estimate):
            estimates.append(estimate)
    if not estimates:
        return {"lower": float("nan"), "upper": float("nan"), "replicates": 0}
    return {
        "lower": float(np.percentile(estimates, 2.5)),
        "upper": float(np.percentile(estimates, 97.5)),
        "replicates": len(estimates),
    }


# ---------------------------------------------------------------------------
# Windowing and splitting. Identical rules to the frozen pipeline.
# ---------------------------------------------------------------------------
def build_windows(df):
    features, labels, rows = [], [], []
    grouped = df.groupby(["global_participant_id", "condition"], sort=False)
    for (participant, condition), session in grouped:
        session = session.sort_values("elapsed_s").reset_index(drop=True)
        session_features = session[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        session_fms = session["fms"].to_numpy(dtype=np.float32)
        source = str(session["source_dataset"].iloc[0])
        for start in range(0, len(session) - WINDOW_LENGTH + 1, WINDOW_STRIDE):
            end = start + WINDOW_LENGTH
            features.append(session_features[start:end])
            labels.append(float(session_fms[start:end].mean()))
            rows.append((str(participant), str(condition), source, start, end))
    metadata = pd.DataFrame(
        rows,
        columns=[
            "participant_id",
            "condition",
            "source_dataset",
            "start_index",
            "end_index",
        ],
    )
    return np.stack(features).astype(np.float32), np.asarray(labels, np.float32), metadata


def participant_strata(labels, metadata):
    """Source folder crossed with a within-folder severity tier."""
    elevated = (labels > ELEVATED_THRESHOLD).astype(int)
    severity = (
        pd.DataFrame(
            {"participant_id": metadata["participant_id"], "elevated": elevated}
        )
        .groupby("participant_id")["elevated"]
        .mean()
        .reset_index(name="elevated_rate")
    )
    info = (
        metadata.groupby("participant_id")["source_dataset"]
        .first()
        .reset_index()
        .merge(severity, on="participant_id")
    )
    info["rank"] = info.groupby("source_dataset")["elevated_rate"].rank(method="first")
    info["severity_tier"] = info.groupby("source_dataset")["rank"].transform(
        lambda values: pd.qcut(values, 2, labels=["low", "high"])
    )
    info["stratum"] = info["source_dataset"] + "_" + info["severity_tier"].astype(str)
    return info


def stratified_participant_split(info, seed):
    """Draw a fresh train/validation/test assignment for one composition.

    Proportions match the frozen split: 20 % test, then 12.5 % of the
    remainder to validation.
    """
    rng = np.random.default_rng(seed)
    train_ids, validation_ids, test_ids = [], [], []
    for _, group in info.groupby("stratum", sort=True):
        members = group["participant_id"].to_numpy()
        members = members[rng.permutation(len(members))]
        n_test = max(1, int(round(0.20 * len(members))))
        n_validation = max(1, int(round(0.125 * (len(members) - n_test))))
        test_ids.extend(members[:n_test])
        validation_ids.extend(members[n_test : n_test + n_validation])
        train_ids.extend(members[n_test + n_validation :])
    return set(train_ids), set(validation_ids), set(test_ids)


def standardize(x, train_index, other_indices):
    train_flat = x[train_index].reshape(-1, x.shape[-1]).astype(np.float64)
    mean = train_flat.mean(axis=0)
    scale = train_flat.std(axis=0)
    scale[scale == 0] = 1.0

    def transform(index):
        return ((x[index].astype(np.float64) - mean) / scale).astype(np.float32)

    return transform(train_index), [transform(i) for i in other_indices]


# ---------------------------------------------------------------------------
# Model. The same CNN used as the base of the frozen five-model ensemble.
# ---------------------------------------------------------------------------
def build_cnn_classifier(tf):
    layers = tf.keras.layers
    inp = layers.Input(shape=(WINDOW_LENGTH, len(FEATURE_COLUMNS)))
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(8, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out)


def balanced_class_weights(binary_labels):
    counts = np.bincount(binary_labels, minlength=2).astype(float)
    counts[counts == 0] = 1.0
    total = float(len(binary_labels))
    return {0: total / (2.0 * counts[0]), 1: total / (2.0 * counts[1])}


def run_one_composition(tf, x, labels, metadata, seed, epochs=100):
    """Full pipeline for one participant assignment; returns a result dict."""
    info = participant_strata(labels, metadata)
    train_ids, validation_ids, test_ids = stratified_participant_split(info, seed)

    participant = metadata["participant_id"]
    is_seated = metadata["source_dataset"].isin(PASSIVE_SOURCES)

    train_index = np.flatnonzero(participant.isin(train_ids))
    validation_index = np.flatnonzero(participant.isin(validation_ids))
    test_index = np.flatnonzero(participant.isin(test_ids) & is_seated)

    binary = (labels > ELEVATED_THRESHOLD).astype(int)
    y_test = binary[test_index]
    if y_test.sum() == 0 or y_test.sum() == len(y_test):
        return {
            "composition_seed": int(seed),
            "status": "skipped_single_class_test",
            "test_windows": int(len(test_index)),
            "test_participants": int(metadata.iloc[test_index]["participant_id"].nunique()),
        }

    x_train, (x_validation, x_test) = standardize(
        x, train_index, [validation_index, test_index]
    )
    y_train = binary[train_index]
    y_validation = binary[validation_index]
    weights = balanced_class_weights(y_train)

    member_scores = []
    for member_seed in ENSEMBLE_SEEDS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(member_seed)
        model = build_cnn_classifier(tf)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
            loss="binary_crossentropy",
            metrics=[tf.keras.metrics.AUC(name="auc")],
        )
        model.fit(
            x_train,
            y_train,
            validation_data=(x_validation, y_validation),
            class_weight=weights,
            epochs=epochs,
            batch_size=32,
            verbose=0,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_auc",
                    mode="max",
                    patience=12,
                    min_delta=1e-4,
                    restore_best_weights=True,
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor="val_auc", mode="max", factor=0.5, patience=5, min_lr=1e-6
                ),
                tf.keras.callbacks.TerminateOnNaN(),
            ],
        )
        member_scores.append(model.predict(x_test, verbose=0).reshape(-1))

    ensemble = np.mean(np.stack(member_scores), axis=0)
    participant_ids = metadata.iloc[test_index]["participant_id"].to_numpy()
    interval = cluster_bootstrap_auc(
        y_test, ensemble, participant_ids, BOOTSTRAP_REPLICATES, seed
    )

    per_participant = []
    for pid in np.unique(participant_ids):
        rows = participant_ids == pid
        per_participant.append(
            {
                "participant_id": str(pid),
                "windows": int(rows.sum()),
                "elevated": int(y_test[rows].sum()),
                "auc": roc_auc(y_test[rows], ensemble[rows]),
            }
        )

    return {
        "composition_seed": int(seed),
        "status": "complete",
        "test_participants": int(len(np.unique(participant_ids))),
        "test_windows": int(len(test_index)),
        "elevated_windows": int(y_test.sum()),
        "prevalence": float(y_test.mean()),
        "pooled_auc": roc_auc(y_test, ensemble),
        "average_precision": average_precision(y_test, ensemble),
        "cluster_ci_lower": interval["lower"],
        "cluster_ci_upper": interval["upper"],
        "cluster_replicates": interval["replicates"],
        "member_aucs": [roc_auc(y_test, s) for s in member_scores],
        "per_participant": per_participant,
        "train_participants": int(len(train_ids)),
        "validation_participants": int(len(validation_ids)),
    }


def summarize(records):
    """Distributional summary only. No significance test: see module docstring."""
    complete = [r for r in records if r.get("status") == "complete"]
    aucs = np.asarray([r["pooled_auc"] for r in complete], dtype=float)
    above = [r for r in complete if r["cluster_ci_lower"] > 0.5]
    return {
        "n_compositions_attempted": len(records),
        "n_compositions_complete": len(complete),
        "mean_auc": float(aucs.mean()),
        "sd_auc": float(aucs.std(ddof=1)),
        "median_auc": float(np.median(aucs)),
        "min_auc": float(aucs.min()),
        "max_auc": float(aucs.max()),
        "n_with_ci_above_chance": len(above),
        "n_with_point_estimate_above_chance": int(np.sum(aucs > 0.5)),
        "note": (
            "Compositions share one pool of 84 participants and are therefore "
            "dependent. Report the distribution; do not apply a one-sample "
            "test against 0.5 or describe the compositions as independent."
        ),
    }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_frame = pd.read_csv(DATASET_PATH)
X_ALL, LABELS_ALL, META_ALL = build_windows(_frame)
print('windows =', len(LABELS_ALL),
      '| participants =', META_ALL["participant_id"].nunique(),
      '| elevated =', float((LABELS_ALL > ELEVATED_THRESHOLD).mean()))
print('setup complete')


In [ ]:
# ============================================================
# CELL 2 - SMOKE TEST.  One composition, 3 epochs.  ~1 minute.
# Continue only if this prints "Smoke test passed".
# ============================================================
_probe = run_one_composition(tf, X_ALL, LABELS_ALL, META_ALL, seed=0, epochs=3)
print(json.dumps({k: v for k, v in _probe.items() if k != 'per_participant'}, indent=2))
assert _probe['status'] == 'complete', _probe
assert 0.0 <= _probe['pooled_auc'] <= 1.0
assert _probe['test_participants'] > 0
print('Smoke test passed')


In [ ]:
# ============================================================
# CELL 3 - FULL RUN.  20 compositions x 5 ensemble members.
# Safe to re-run: finished compositions are detected and skipped.
# ============================================================
records = []
for seed in COMPOSITION_SEEDS:
    path = OUTPUT_DIR / f'composition_{seed:03d}.json'
    if path.exists():
        print(f'[resume] composition {seed} already complete')
        records.append(json.loads(path.read_text()))
        continue
    print(f'[run] composition {seed}', flush=True)
    record = run_one_composition(tf, X_ALL, LABELS_ALL, META_ALL, seed)
    path.write_text(json.dumps(record, indent=2) + '\n')
    records.append(record)
    print(f"      {record['status']} auc={record.get('pooled_auc')}", flush=True)

summary = summarize(records)
(OUTPUT_DIR / 'composition_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('per_participant', 'member_aucs')}
    for r in records
]).to_csv(OUTPUT_DIR / 'composition_results.csv', index=False)
print(json.dumps(summary, indent=2))
